<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day21_practice4_%EC%8B%A4%EC%A0%84RAG_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_SDK__LCEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# SDK (Software Development Kit)는 프로그램을 쉽게 개발할 수 있도록 제공하는 개발 종합 키트
# API (Application Programming Interface) 는 프로그램 간에 데이터를 주고받기 위한 규칙

knowledge/ 폴더 (PDF·txt·md)
   → ① 로드(pypdf)  → ② 청킹  → ③ 임베딩(Gemini)  → ④ 벡터DB(Chroma)
   → ⑤ retriever  → ⑥ LCEL 체인(검색+생성)  → chain.invoke("질문")

In [2]:
!pip install -q langchain langchain-core langchain-openai langchain-chroma langchain-text-splitters pypdf langchain-google-genai

In [3]:
import os, requests

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get('VISION_API_KEY')
  except Exception:
    return os.environ.get("VISION_API_KEY","") # Colab이 아니거나 Secret 이 없을 때 환경변수 딕셔너리에서 꺼냄
USE_REAL_AI = bool(_get_api_key()) # 키 있으면 True(실제 AI) 없으면 False(mock)
FALLBACK_ON_ERROR = True  # real 이 실패해도 mock 으로 앱 실행.

API_KEY = _get_api_key()
# API 호출 - openAI 호환 형식으로 작성
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai" # Gemini OpenAI 호환

In [4]:
# 1. 문서 로드 : PDF 는 pypdf 로 페이지별 텍스트 추출
from langchain_core.documents import Document
import glob, os

def load_documents(folder="/content/drive/MyDrive/kwu/DL/knowledge"):
  docs = []
  for path in sorted(glob.glob(os.path.join(folder, "*"))): # path: 파일 하나의 전체 경로
    name = os.path.basename(path) # 파일명
    if path.endswith(".pdf"):
      from pypdf import PdfReader # 함수/조건문 안에서도 import 가능
      reader = PdfReader(path) # pages 속성에 페이지들이 들어 있음
      for i, page in enumerate(reader.pages): # 페이지번호, 해당 페이지 한 장
        text = page.extract_text() # 글자만 문자열로, 스캔 이미지 PDF면 "" 빈 문자열
        if text and text.strip(): # A and B일 때 A가 False면 B는 안 본다.
          docs.append(Document(page_content=text, metadata={"source": name, "page": i}))
    elif path.endswith({".txt",".md"}):
      with open(path, encoding="utf-8") as f:
        text = f.read().strip()
      if text:
        docs.append(Document(page_content=text, metadata={"source":name}))
  return docs

documents = load_documents()
print(f"1 로드: 문서 {len(documents)}개")
for d in documents:
  print(f"  [{d.metadata["source"]}] {len(d.page_content)}자")



1 로드: 문서 5개
  [01_도로법_포트홀.pdf] 108자
  [02_맨홀_지침.pdf] 83자
  [03_균열_기준.pdf] 85자
  [04_신고_절차.pdf] 73자
  [05_종합_도로관리_규정.pdf] 632자


In [5]:
# 2. 청킹 : 긴 문서를 검색 단위로 자른다
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"2. 청킹: 문서 {len(documents)}개 → 청크 {len(chunks)}개")

2. 청킹: 문서 5개 → 청크 6개


In [6]:
# 3. 임베딩 + 벡터DB - Gemini 임베딩을 Chroma 벡터DB에
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 설정 (Gemini)
# OpenAIEmbeddings 에 base_url만 Gemini로 바꾸면 Gemini 임베딩이 쓰인다 (OpenAI 호환)
embeddings = OpenAIEmbeddings(
    model="gemini-embedding-2", # 429
    base_url=GEMINI_BASE_URL,
    api_key=API_KEY,
    check_embedding_ctx_length=False, # OpenAI 전용 토큰검사 끝
)

# 벡터 DB 구축 (Chroma)
vectorstore = Chroma.from_documents(  # 청크들을 임베딩해서 벡터DB에 적재
  documents=chunks,
  embedding=embeddings,
  collection_name="road_regulations",
  persist_directory="/content/drive/MyDrive/kwu/DL/chroma_db"   # 디스크에 영속 저장
)
print(f" 3. 벡터 DB 구축 완료: 청크 {len(chunks)}개 적재 (Gemini 임베딩)")

 3. 벡터 DB 구축 완료: 청크 6개 적재 (Gemini 임베딩)


In [7]:
# 셀 4. retriever + LCEL 체인 : 검색+생성을 파이프로
from langchain_openai import ChatOpenAI         # OpenAI 호환 API 규격으로 LLM을 호출
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser # LLM 응답은 AIMessage 객체 → str 변환
from langchain_core.runnables import RunnablePassthrough # 입력값(질문)을 그대로 다음 단계로 보내는
from langchain_google_genai import ChatGoogleGenerativeAI # Gemini Chat 모델 사용

# retriever - 벡터DB를 '검색기' 객체로, k=3: 상위 3개 청크
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# llm = ChatOpenAI(model="gemini-pro", base_url=GEMINI_BASE_URL, api_key=API_KEY, temperature=0.2) # (Original Line - commented out)
llm = ChatGoogleGenerativeAI(model="gemini-embedding-2", google_api_key=API_KEY, temperature=0.2)

# 프롬포트 - 검색된 규정을 근거로만 답하게
prompt = ChatPromptTemplate.from_template(
    """ 너는 도로 관리 규정 안내 담당자다. 아래 [규정]에 근거해서만 답하라.
    규정에 없으면 '규정에 없습니다'라고 답하라, 근거 조항을 인용하며 두 문장 이내로,

    [규정]
    {context}

    [질문] {question}
    [답변]
    """
)

def format_docs(docs):
  return "\n\n".join(d.page_content for d in docs) # 한 덩어리로

# LCEL 체인 - 검색+생성을 파이프(|)로 연결
# LangChain에서 여러 구성 요소(프롬포트, 모델, 출력 파서 등)를 | (파이프) 연산자를 사용해 직관적이고 표준화된 방식으로 연결(체이닝)할 수 있도록
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)
# 질문으로 검색 → 청크를 문자열로 # 질문은 그대로 통과 # 규정+질문을 프롬포트에 채움 # LLM이 답 생성 # 답에서 문자열만 추출
print("셀 4. RAG 체인 완성 - rag_chain.invoke('질문') 한 줄로 검색+생성!")

셀 4. RAG 체인 완성 - rag_chain.invoke('질문') 한 줄로 검색+생성!


In [8]:
# 셀 5. top-k 검색 결과 확인 - 무엇을 근거로 답하나
test_q = "심각도 4 포트홀은 언제 처리되나요"
hits = retriever.invoke(test_q)

print(f"질문: {test_q}")
print(f"검색된 청크: {len(hits)}개 (k=3)\n")

for rank, doc in enumerate(hits, 1):
  src = doc.metadata.get("source", "?")
  page = doc.metadata.get("page", "")
  preview = doc.page_content[:150].replace("\n", "")
  print(f"[{rank}위] 출처: {src}" + (f"(p.{page+1})" if page != "" else ""))
  print(f"내용: {preview}...\n")

질문: 심각도 4 포트홀은 언제 처리되나요
검색된 청크: 3개 (k=3)

[1위] 출처: 01_도로법_포트홀.pdf(p.1)
내용: 01 도로법 포트홀도로법 시행령 제55조에 따라 도로 관리청은 포트홀 등 도로 파손 발견 시 24시간 이내 응급 복구 조치를 시행해야 한다. 심각도 4 이상은 긴급 보수팀을 우선 배정한다....

[2위] 출처: 01_도로법_포트홀.pdf(p.1)
내용: 01 도로법 포트홀도로법 시행령 제55조에 따라 도로 관리청은 포트홀 등 도로 파손 발견 시 24시간 이내 응급 복구 조치를 시행해야 한다. 심각도 4 이상은 긴급 보수팀을 우선 배정한다....

[3위] 출처: 01_도로법_포트홀.pdf(p.1)
내용: 01 도로법 포트홀도로법 시행령 제55조에 따라 도로 관리청은 포트홀 등 도로 파손 발견 시 24시간 이내 응급 복구 조치를 시행해야 한다. 심각도 4 이상은 긴급 보수팀을 우선 배정한다....



In [9]:
# 셀 6. 실행 - 질문 하나로 검색+답변
질문들 = ["포트홀은 얼마나 빨리 복구해야 하나요?",
          "맨홀 뚜껑이 빠졌을 때 어떻게 하나요?",
          "신고는 어디로 하나요?"]
if not API_KEY:
  print("키가 없어 실제 답변을 못 만듭니다. Colab Secrets 에 VISION_API_KEY 를 넣으세요.")
else:
  for q in 질문들:
    print(f"질문: {q}")
    answer = rag_chain.invoke(q)
    print(f"답변: {answer}")

질문: 포트홀은 얼마나 빨리 복구해야 하나요?


GoogleModelNotFoundError: Error calling model 'gemini-embedding-2' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-embedding-2 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}